# **C11 - Monte Carlo y Error en el mismo**
## **Simulación de Procesos Financieros**

---

**Autor:** Francisco Uriel Ledezma Chávez  
**Carrera:** Ingeniería Financiera  
**Institución:** ITESO — Universidad Jesuita de Guadalajara  
**Semestre:** 5° — Simulación de Procesos Financieros  
**Fecha de realización:** 30 de septiembre de 2026  

---

**Ejercicio:** valuar una call europea con $S_0 = 100$, $K = 100$, $r = 5\%$, $\sigma = 20\%$ y $T = 1$ mediante Monte Carlo con 10, 100, 1,000, 10,000, 100,000 y 1,000,000 simulaciones, comparar cada estimación contra la fórmula de Black-Scholes y repetir después el experimento 50 veces por tamaño de muestra para promediar los errores y revisar cómo disminuyen conforme crece la simulación.

Los notebooks anteriores reportaron el error estándar junto a cada precio simulado, aunque siempre sobre un solo tamaño de muestra, mientras que aquí la pregunta es cómo se comporta ese error cuando el número de simulaciones crece en potencias de diez, y la respuesta tiene una forma precisa, ya que el error de Monte Carlo no cae en proporción a $M$ sino a $\sqrt{M}$, de modo que cada decimal adicional de precisión cuesta cien veces más simulaciones.

## Índice de contenidos

1. Sección 1 — Precio terminal y forma cerrada
2. Sección 2 — Convergencia del precio en una sola corrida
3. Sección 3 — Distribución del error sobre 50 repeticiones
4. Sección 4 — Conclusiones

## Importación de bibliotecas

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

np.random.seed(30)

## 1. Precio terminal y forma cerrada

La call europea sólo depende del precio al vencimiento, por lo cual basta un salto único del movimiento browniano geométrico en lugar de la trayectoria completa que exigían las opciones asiáticas y de barrera, y `precio_final` genera de una sola vez los $M$ precios terminales.

$$S_T = S_0 \exp\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z\right], \qquad Z \sim N(0,1)$$

La referencia contra la cual se mide el error es la fórmula de Black-Scholes, que para este contrato entrega el precio exacto bajo los mismos supuestos de la simulación, de modo que cualquier diferencia entre ambas cifras proviene únicamente del muestreo y no de una discrepancia entre modelos.

$$C = S_0 N(d_1) - K e^{-rT} N(d_2), \qquad d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

In [2]:
def precio_final(S0, r, sigma, T, M):
    Z = np.random.standard_normal(M)
    return S0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)

def d1_d2(S, K, r, sigma, T, q=0):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return d1, d1 - sigma * np.sqrt(T)

def bs(S, K, r, sigma, T, call=True, q=0):
    d1, d2 = d1_d2(S, K, r, sigma, T, q)
    if call:
        return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)

El precio de Monte Carlo es el pago promedio descontado a la tasa libre de riesgo, y `valor_presente` lo devuelve junto con su error estándar, que mide la dispersión esperada de ese promedio alrededor del valor verdadero, donde $s$ es la desviación estándar muestral de los $M$ pagos.

$$\hat{C} = e^{-rT}\,\frac{1}{M}\sum_{i=1}^{M}\max\left(S_T^{(i)} - K,\ 0\right), \qquad \text{EE} = e^{-rT}\,\frac{s}{\sqrt{M}}$$

In [3]:
def payoff_call(ST, K, call = True):
    if call:
        return np.maximum(ST - K, 0)
    else:
        return np.maximum(K - ST, 0)

def valor_presente(payoffs, r, T):
    desc = np.exp(-r * T)
    return float(desc * np.mean(payoffs)), float(desc * np.std(payoffs, ddof=1) / np.sqrt(len(payoffs)))

## 2. Convergencia del precio en una sola corrida

Cada tamaño de muestra se simula una sola vez, y la tabla coloca junto al precio de Monte Carlo el de Black-Scholes, el error estándar que la propia simulación estima y el error absoluto efectivamente observado frente a la forma cerrada.

In [4]:
p10 = precio_final(100, 0.05, 0.2, 1, 10)
p100 = precio_final(100, 0.05, 0.2, 1, 100)
p1000 = precio_final(100, 0.05, 0.2, 1, 1000)
p10000 = precio_final(100, 0.05, 0.2, 1, 10000)
p100000 = precio_final(100, 0.05, 0.2, 1, 100000)
p1000000 = precio_final(100, 0.05, 0.2, 1, 1000000)

Price = bs(100, 100, 0.05, 0.2, 1)
print(f"Price: {Price}")

po10 = payoff_call(p10, 100, call=True)
po100 = payoff_call(p100, 100, call=True)
po1000 = payoff_call(p1000, 100, call=True)
po10000 = payoff_call(p10000, 100, call=True)
po100000 = payoff_call(p100000, 100, call=True)
po1000000 = payoff_call(p1000000, 100, call=True)

error10 = valor_presente(po10, 0.05, 1)
error100 = valor_presente(po100, 0.05, 1)
error1000 = valor_presente(po1000, 0.05, 1)
error10000 = valor_presente(po10000, 0.05, 1)
error100000 = valor_presente(po100000, 0.05, 1)
error1000000 = valor_presente(po1000000, 0.05, 1)

df = pd.DataFrame({
    "M": [10, 100, 1000, 10000, 100000, 1000000],
    "Monte Carlo": [error10[0], error100[0], error1000[0], error10000[0], error100000[0], error1000000[0]],
    "Black-Scholes": [Price] * 6,
    "Error estándar": [error10[1], error100[1], error1000[1], error10000[1], error100000[1], error1000000[1]],
    "Error absoluto": [abs(e[0] - Price) for e in [error10, error100, error1000, error10000, error100000, error1000000]]
})

df

Price: 10.450583572185565


,M,Monte Carlo,Black-Scholes,Error estándar,Error absoluto
0,10,10.550658,10.450584,4.892484,0.100075
1,100,9.933801,10.450584,1.318678,0.516783
2,1000,10.484972,10.450584,0.471264,0.034389
3,10000,10.499202,10.450584,0.145172,0.048618
4,100000,10.395638,10.450584,0.046456,0.054945
5,1000000,10.454828,10.450584,0.014744,0.004244


La forma cerrada fija el precio de la call en 10.4506, y la simulación se acerca a esa cifra de manera desigual, ya que con 10 simulaciones entrega 10.5507 y con 100 cae a 9.9338, mientras que a partir de 1,000 se mantiene entre 10.3956 y 10.4992 hasta cerrar en 10.4548 con un millón de simulaciones, apenas 0.0042 por encima del valor exacto.

El error absoluto de una sola corrida no es una buena medida de la precisión, pues con 10 simulaciones resulta de apenas 0.1001, menor que el de 100,000 simulaciones, que alcanza 0.0549, y esa inversión no indica que la muestra pequeña sea mejor sino que acertó por azar, como deja claro su error estándar de 4.8925, casi la mitad del precio que pretende estimar.

El error estándar sí desciende de forma ordenada, de 4.8925 a 1.3187, 0.4713, 0.1452, 0.0465 y 0.0147, con cocientes entre filas consecutivas que oscilan entre 2.80 y 3.71 y se estabilizan alrededor de 3.15 en las muestras grandes, cifra cercana a $\sqrt{10} \approx 3.16$, lo cual confirma que multiplicar las simulaciones por diez reduce la incertidumbre del precio apenas a poco menos de un tercio.

## 3. Distribución del error sobre 50 repeticiones

Una sola corrida no permite separar la precisión del método de la suerte de la muestra, por lo cual `errores` repite la valuación `n` veces para cada tamaño `simulaciones`, generando en cada repetición una muestra nueva de precios terminales, descontando su pago promedio y midiendo la distancia absoluta contra el precio de Black-Scholes, para devolver al final el promedio y la desviación de esos 50 errores.

El enunciado pide el experimento hasta 10,000 simulaciones, y aquí se extiende hasta 1,000,000 para conservar la misma malla de la sección anterior y comprobar que la tendencia se sostiene dos órdenes de magnitud más allá.

In [5]:
def errores(n:int, simulaciones):
    error_array = np.zeros(n)
    for i in range(len(error_array)):
        p = precio_final(S0=100, r=0.05, sigma=0.2, T=1, M=simulaciones)
        po = payoff_call(p, K=100, call=True)
        error = abs(valor_presente(po, r=0.05, T=1)[0] - Price)
        error_array[i] = error
    return error_array.mean(), error_array.std()

In [6]:
e10m, e10s = errores(n=50, simulaciones=10)
e100m, e100s = errores(n=50, simulaciones=100)
e1000m, e1000s = errores(n=50, simulaciones=1000)
e10000m, e10000s = errores(n=50, simulaciones=10000)
e100000m, e100000s = errores(n=50, simulaciones=100000)
e1000000m, e1000000s = errores(n=50, simulaciones=1000000)

df_e = pd.DataFrame({'M': [10, 100, 1000, 10000, 100000, 1000000], 
                     'Error medio': [e10m, e100m, e1000m, e10000m, e100000m, e1000000m], 
                     'Desviación del error': [e10s, e100s, e1000s, e10000s, e100000s, e1000000s]})

df_e

,M,Error medio,Desviación del error
0,10,3.265656,2.456305
1,100,1.108861,1.042131
2,1000,0.351015,0.226961
3,10000,0.109027,0.085330
4,100000,0.035695,0.023866
5,1000000,0.010832,0.008075


El error medio cae de 3.2657 con 10 simulaciones a 1.1089 con 100, 0.3510 con 1,000, 0.1090 con 10,000, 0.0357 con 100,000 y 0.0108 con un millón, de modo que cada multiplicación por diez divide el error entre un factor de 2.95 a 3.30, y entre el primer y el último renglón la reducción acumulada es de 301.5 veces, muy cerca de las $\sqrt{10^5} \approx 316.2$ que predice la ley de convergencia $1/\sqrt{M}$.

A diferencia de la sección anterior, aquí el descenso es monótono, ya que promediar 50 repeticiones elimina la suerte de una corrida aislada y deja ver únicamente la precisión del método, mientras que la desviación del error decrece al mismo ritmo, de 2.4563 a 0.0081, con lo cual no sólo se reduce el error típico sino también su variabilidad entre repeticiones.

Las cifras concuerdan con la teoría en nivel y no sólo en pendiente, pues si el precio estimado se distribuye de forma aproximadamente normal alrededor del valor exacto, el error absoluto esperado equivale a $\sqrt{2/\pi} \approx 0.80$ veces el error estándar, y con 10,000 simulaciones el error estándar de 0.1452 de la sección anterior anticipa así un error medio de 0.1158 frente a los 0.1090 observados, diferencia que cabe dentro del margen que dejan 50 repeticiones.

## 4. Conclusiones

El ejercicio confirma que Monte Carlo converge al precio de Black-Scholes, aunque con lentitud, ya que el error medio de 50 repeticiones pasa de 3.2657 con 10 simulaciones a 0.0108 con un millón, y cada multiplicación por diez del tamaño de muestra apenas lo divide entre un factor cercano a $\sqrt{10} \approx 3.16$.

La explicación está en que el precio simulado es un promedio, y la dispersión de un promedio de $M$ observaciones independientes decrece con $\sqrt{M}$ y no con $M$, de modo que el error estándar que estima una sola corrida anticipa correctamente tanto la pendiente como el nivel del error que se observa al repetir el experimento.

La primera tabla deja además una advertencia práctica, ya que una corrida aislada puede acertar por azar, como ocurrió con 10 simulaciones y un error de apenas 0.1001, y por ello la precisión de un precio de Monte Carlo debe leerse en su error estándar y no en la cercanía casual de una sola estimación a la cifra exacta.

La consecuencia es directa, pues ganar un decimal de precisión exige cien veces más simulaciones y con un millón de ellas el error ronda todavía el centavo, de manera que para contratos sin forma cerrada conviene reportar siempre el error estándar junto al precio y, cuando el costo de cómputo se vuelve una restricción, recurrir a técnicas de reducción de varianza antes que aumentar indefinidamente el tamaño de la muestra.

El experimento tiene un límite que conviene declarar, ya que cada error medio se calcula sobre sólo 50 repeticiones y arrastra por ello una incertidumbre propia cercana al 10% de su valor, suficiente para confirmar la tendencia pero no para leer las diferencias entre cocientes vecinos, de modo que la extensión natural es repetir el experimento con más repeticiones y graficar el error contra $M$ en escala logarítmica, donde la ley de convergencia aparece como una recta de pendiente $-1/2$.